## The performance of a lock complex
In this notebook, we simulate a lock object on a network. Randomly generated, evenly sized vessels sampled from an exponential distribution have to pass this lock. We add a complex lock object to the graph. Vessels are levelled in the same lock operation if they can fit inside the lock, and register with the lock operator before the start of the lock operation. Based on this behaviour, we will quantify the performance of the lock in terms of efficiency and capacity.

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.core.utils import create_object
from opentnsim.utils import inspect_object, generate_vessels_from_distribution
from opentnsim.graph import mixins as graph_module
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core import Identifiable, Movable, VesselProperties, ExtraMetadata
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from opentnsim.graph.mixins import HasMultiDiGraph
from opentnsim.output import HasOutput
from scipy.stats import norm, uniform, expon

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable
from opentnsim.lock.calculations import estimate_lock_capacity, calculate_lock_occupancy
from opentnsim.lock.logutils import calculate_cycle_information, get_vessel_delays
from opentnsim.lock.visualizations import show_results

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.4


#### 0. Create environment

In [2]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

#### 1. Create graph
We create a directional graph with a layout similar to that of notebooks 0201, 0202, and 0203. However, we add two additional edges prior to nodes 0 and 1 to show that a lock can also be implemented in a larger graph. These edges are 20km long.

In [3]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

# create a directed graph
graph = nx.DiGraph()

# add nodes
graph.add_node('-1',geometry=transform(wgs84eqd_to_wgs84rad,Point(-25000,0)))
graph.add_node('0',geometry=transform(wgs84eqd_to_wgs84rad,Point(-5000,0)))
graph.add_node('1',geometry=transform(wgs84eqd_to_wgs84rad,Point(5000,0)))
graph.add_node('+1',geometry=transform(wgs84eqd_to_wgs84rad,Point(25000,0)))

# add edges
graph.add_edge('-1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-25000, 0),Point(-5000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('0','-1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(-25000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('0','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(-5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','+1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(25000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('+1','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(25000, 0),Point(5000, 0)])), weight=1, length_m=25000-5000)

# add graph to environment
env.graph = graph

In [4]:
graph_module.plot_graph(graph)

#### 1+ Adding infrastructure
We construct the same lock complex as before.

In [5]:
lock_chamber = IsLockChamber(env=env,
                             lock_length = 400,
                             lock_width = 50,
                             lock_depth = 10,
                             name='Lock',
                             gate_open = '0',
                             edge = ('0','1'),
                             geometry_m = Polygon([Point(-200, -25),Point(-200, 25),Point(200, 25),Point(200, -25)]))

In [6]:
# The minimum required input for a lock complex are waiting areas at both sides of the lock
waiting_area_A = IsLockWaitingArea(env=env,
                                   name = 'Waiting area A',
                                   edge = ('0','1'),
                                   distance_from_edge_start = 0)

waiting_area_B = IsLockWaitingArea(env=env,
                                   name = 'Waiting area B',
                                   edge = ('1','0'),
                                   distance_from_edge_start = 0)

In [7]:
lock_complex = IsLockComplex(lock_chambers = [lock_chamber],
                             waiting_areas = [waiting_area_A, waiting_area_B],
                             registration_nodes = ['0','1'],
                             env=env,
                             name = 'Lock complex',)

#### 2. Create agents
We create the same vessel agent object.

In [8]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
        Identifiable,               # allows to give the object a name and a random ID,
        Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        HasMultiDiGraph,            # allow to operate on a graph that can include parallel edges from and to the same nodes
        HasOutput,                  # allow additional output to be stored
    ), 
)

In [9]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

To better illustrate the lock's performance, we add two vessel generators at the network boundaries. From each direction, 10 equally sized vessels are generated randomly from an exponential distribution with a mean arrival rate of 30 minutes. We use different seeds to get different arrival times.

In [10]:
upstream_vessels = generate_vessels_from_distribution(env=env,
                                                      VesselClass = Vessel,
                                                      vessel_parameters = {'v':4, 'L':100, 'B':20, 'T':5, 'type':'tanker'},
                                                      mean_arrival_rate=30.,
                                                      number_of_vessels=10,
                                                      start_node = '-1',
                                                      end_node = '+1',
                                                      seed = 123)

downstream_vessels = generate_vessels_from_distribution(env=env,
                                                        VesselClass = Vessel,
                                                        vessel_parameters = {'v':4, 'L':100, 'B':20, 'T':5, 'type':'tanker'},
                                                        mean_arrival_rate=30.,
                                                        number_of_vessels=10,
                                                        start_node = '+1',
                                                        end_node = '-1',
                                                        seed = 456)

vessels = upstream_vessels + downstream_vessels

for vessel in vessels:
    env.process(mission(env, vessel))

#### 3. Run simulation
We run the simulation.

In [11]:
env.run()

#### 4. Inspect output
We skip the logbooks and inspect the Gantt charts and time-distance diagram.

##### Gantt chart of event table
The Gantt chart becomes quite full

In [12]:
df_eventtable = opentnsim.core.logutils.logbook2eventtable([*vessels, lock_chamber])
fig = generate_vessel_gantt_chart(df_eventtable)

##### Time-distance diagram of vessels passing the lock and planning info
Luckily, the time-distance diagram shows the behaviour of the vessels in a more comprehensible manner. Waiting times emerge.

In [13]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_chamber.plot(xlimmin = -6050, 
                        xlimmax = 6050,
                        ylimmin = pd.Timestamp('2025-01-01 00:00:00'),
                        ylimmax = pd.Timestamp('2025-01-01 10:00:00'),
                        method='Plotly',
                        boundary_nodes = ['-1','+1'])

fig.update_layout(height=cm_to_pixels(20))

##### Lock performance
We can simply get the performance using the <i>get_performance</i> function of the <strong>lock chamber</strong> object. This function calculates various KPI (key performance indicators):
- <strong>Vessel delay</strong>:
we calculate observed delays of vessels relative to their normal cruising speed at the edge. Delays emerge due to waiting times in the waiting areas and at the lock, levelling times, and sailing in and out of the lock at reduced speeds.
- <strong>Lock occupancy</strong>:
we calculate the lock occupancy by dividing the sum of the lengths of the vessels by the length dimension of the lock chamber.
- <strong>I/C ratio</strong> (traffic load, see Section 3.1 of Lecture Notes):
we calculate the intensity and divide it by the estimated lock capacity, both expressed as the number of vessels that pass the lock per hour. The intensity is calculated by identifying the lock operation cycles, their durations and the number of levelled vessels. The capacity is calculated based on the operation times, including loop times, sailing in times and gaps between vessels, and levelling times (see example box 3.1 of the Lecture Notes).

We start by calculating information for each cycle. We determine each cycle, including how long each lock process took (i.e., start and stop times, directions, loop times, sailing-in times, closing doors, levelling, opening doors, sailing-out times, and how many vessels were levelled).

In [14]:
Tc_df = calculate_cycle_information(lock_chamber)
Tc_df

,Start time of cycle,Stop time of cycle,Direction first operation,Direction second operation,Loop time start side,Sailing-in time start side,Closing gate time start side,Levelling time to opposing side,Opening gate time opposing side,Sailing-out time opposing side,...,Closing gate time opposing side,Levelling time to start side,Opening gate time start side,Sailing-out time start side,Cycle duration,Number of upstream vessels,Number of downstream vessels,Upstream vessel_ids,Downstream vessel_ids,Intensity (I_s)
0,2025-01-01 01:48:04.354414000,2025-01-01 02:47:01.449925,0,1,0 days 00:00:00,0 days 00:00:00,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:00,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:06:49,0 days 00:58:57,0,3,[],"[fea61d21-0063-4006-92be-9717767c8bb4, 72ebba1...",3.053353
1,2025-01-01 02:08:04.354415001,2025-01-01 03:29:43.642151,1,0,0 days 00:00:00,0 days 00:12:08,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:06:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:09:49,0 days 01:21:39,4,3,"[fea61d21-0063-4006-92be-9717767c8bb4, 72ebba1...","[5d7c5515-d961-418d-aba2-7a8dff4f58e6, a967c37...",5.143605
2,2025-01-01 02:47:01.449926001,2025-01-01 03:59:17.411049,0,1,0 days 00:03:05,0 days 00:09:49,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:09:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:49,0 days 01:12:16,4,1,"[5d7c5515-d961-418d-aba2-7a8dff4f58e6, a967c37...",[f18e968d-93f2-4580-91a9-60e4efa218d3],4.151329
3,2025-01-01 03:29:43.642152000,2025-01-01 04:33:13.987723,1,0,0 days 00:03:05,0 days 00:05:40,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:03:49,0 days 01:03:30,2,1,[f18e968d-93f2-4580-91a9-60e4efa218d3],"[c3c5c9ba-2939-4026-852f-8a917cefa201, ae11fca...",2.834389
4,2025-01-01 03:59:17.411050000,2025-01-01 05:11:33.372173,0,1,0 days 00:03:05,0 days 00:07:03,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:03:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:06:49,0 days 01:12:16,2,3,"[c3c5c9ba-2939-4026-852f-8a917cefa201, ae11fca...","[f12ec47c-2964-40b8-899a-6032add53086, a7d6c07...",4.151329
5,2025-01-01 04:36:18.987723000,2025-01-01 05:34:38.372173,1,0,0 days 00:03:05,0 days 00:08:26,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:06:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:00,0 days 00:58:19,0,3,"[f12ec47c-2964-40b8-899a-6032add53086, a7d6c07...",[],3.086257
6,2025-01-01 05:23:51.166654000,2025-01-01 06:10:19.935553,0,1,0 days 00:00:00,0 days 00:00:00,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:00,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:49,0 days 00:46:29,0,1,[],[1224620e-9f9f-4070-b585-16060e1c299a],1.290892
7,2025-01-01 05:43:51.166654000,2025-01-01 06:48:39.320002,1,0,0 days 00:00:00,0 days 00:05:40,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:06:49,0 days 01:04:48,3,1,[1224620e-9f9f-4070-b585-16060e1c299a],"[fe2aaaa7-dda0-4ec9-91be-729e08fbd9a8, ce18ff8...",3.703558
8,2025-01-01 06:10:19.935554000,2025-01-01 07:22:35.896675,0,1,0 days 00:03:05,0 days 00:08:26,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:06:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:03:49,0 days 01:12:16,3,2,"[fe2aaaa7-dda0-4ec9-91be-729e08fbd9a8, ce18ff8...","[95542132-5db2-4ebf-8562-22de4244dda1, 41f1542...",4.151329
9,2025-01-01 06:48:39.320003000,2025-01-01 07:52:09.665575,1,0,0 days 00:03:05,0 days 00:07:03,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:03:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:49,0 days 01:03:30,1,2,"[95542132-5db2-4ebf-8562-22de4244dda1, 41f1542...",[83dbaecb-8f0f-4455-b9a6-346745a3be0e],2.834389


This is the information for the second cycle:

In [15]:
Tc_df.loc[1]

Start time of cycle                                    2025-01-01 02:08:04.354415001
Stop time of cycle                                        2025-01-01 03:29:43.642151
Direction first operation                                                          1
Direction second operation                                                         0
Loop time start side                                                 0 days 00:00:00
Sailing-in time start side                                           0 days 00:12:08
Closing gate time start side                                         0 days 00:05:00
Levelling time to opposing side                                      0 days 00:10:00
Opening gate time opposing side                                      0 days 00:05:00
Sailing-out time opposing side                                       0 days 00:06:49
Loop time opposing side                                              0 days 00:03:05
Sailing-in time opposing side                                    

We can also get the aggregated cycle information

In [16]:
lock_chamber.get_aggregated_cycle_information()

Number of cycles                                                   5
Number of vessels                                                 20
Minimum cycle time                                   0 days 00:46:29
Average cycle time                                   0 days 01:05:24
Maximum cycle time                                   0 days 01:21:39
Total cycle time                                     0 days 10:54:00
Average duration loop time upstream                  0 days 00:01:51
Average fraction loop time upstream (%)                          3.3
Average duration sailing in time upstream            0 days 00:05:04
Average fraction sailing in time upstream (%)                    8.6
Average duration gate closing time upstream          0 days 00:05:00
Average fraction gate closing time upstream (%)                 7.65
Average duration levelling time to downstream        0 days 00:10:00
Average fraction levelling time to downstream (%)              15.29
Average duration gate opening time

###### <strong>Vessel delay</strong>
We present the results of the vessel delay calculations in the following forms: all the components that contributed to the delay, and aggregated per lock complex area and cause. Delays are defined as such that the vessel speed deviates from its normal cruising speed.

In [17]:
vessel_delays, vessel_delay_locations, vessel_delays_causes = get_vessel_delays(lock_chamber)
vessel_delays

,operation_nr,vessel_id,total_delay,waiting time in waiting_area for other vessels (%),waiting time in waiting_area for available operation (%),delay due to sailing to lock gate (%),delay due to sailing to position in lock (%),waiting time for other vessels to sail into lock (%),waiting time for closing doors (%),waiting time for levelling (%),waiting time for opening doors (%),waiting time for other vessels to sail out of lock (%),delay due to sailing out of lock (%),delay due to sailing away from lock (%)
0,1,fea61d21-0063-4006-92be-9717767c8bb4,0 days 00:41:17,0.0,24.22,0.0,10.20,15.68,12.11,24.22,12.11,0.00,1.46,0.0
1,1,72ebba1d-c551-4c4e-bf3d-e07282aa7636,0 days 00:31:40,0.0,1.22,0.0,9.50,16.08,15.79,31.57,15.79,4.36,5.70,0.0
2,1,9dcd529c-7c86-4207-a7f0-86b508dcdd18,0 days 00:27:34,0.0,0.00,0.0,6.55,0.00,18.13,36.27,18.13,10.01,10.91,0.0
3,2,5d7c5515-d961-418d-aba2-7a8dff4f58e6,0 days 01:17:49,0.0,62.79,0.0,5.41,5.32,6.43,12.85,6.43,0.00,0.77,0.0
4,2,a967c37a-3de1-4abe-8bfc-74765674b4af,0 days 01:17:18,0.0,62.55,0.0,3.89,3.57,6.47,12.94,6.47,1.79,2.33,0.0
5,2,4fa1b252-14f4-4295-b154-99ac3e00de3b,0 days 01:12:45,0.0,60.20,0.0,2.48,1.90,6.87,13.75,6.87,3.79,4.13,0.0
6,2,73910002-ec04-4309-8993-fda974483b61,0 days 01:06:31,0.0,56.47,0.0,0.90,0.00,7.52,15.03,7.52,6.22,6.33,0.0
7,3,f18e968d-93f2-4580-91a9-60e4efa218d3,0 days 01:25:02,0.0,70.82,0.0,4.95,0.00,5.88,11.76,5.88,0.00,0.71,0.0
8,4,c3c5c9ba-2939-4026-852f-8a917cefa201,0 days 02:04:16,0.0,78.92,0.0,3.39,1.11,4.02,8.05,4.02,0.00,0.48,0.0
9,4,ae11fca5-ad5a-4bb8-955a-5aacafcd186f,0 days 00:56:43,0.0,53.82,0.0,5.30,0.00,8.82,17.63,8.82,2.43,3.18,0.0


We observe that most delay takes place in the waiting area.

In [18]:
vessel_delay_locations

nr_operations                         10
nr_vessels                            20
min_vessel_delay         0 days 00:27:34
average_vessel_delay     0 days 01:04:44
max_vessel_delay         0 days 02:04:16
total_delay              0 days 21:34:48
waiting_area (%)                   57.48
sailing_to_lock (%)                 4.92
in_lock (%)                        35.09
sailing_from_lock (%)               2.51
dtype: object

This is because most waiting time is related to congestion. In addition, the lock operation takes significant time and is augmented by additional waiting time for other vessels to sail into/out of the lock chamber. The reduced sailing speed for manoeuvring into/out of the lock contributes to this.

In [19]:
vessel_delays_causes

nr_operations                         10
nr_vessels                            20
min_vessel_delay         0 days 00:27:34
average_vessel_delay     0 days 01:04:44
max_vessel_delay         0 days 02:04:16
total_delay              0 days 21:34:48
congestion (%)                     57.48
obstruction (%)                     7.43
traffic (%)                          4.2
operation of lock (%)              30.89
dtype: object

###### <strong>Lock occupancy</strong>
We show the results of the calculation of the occupancy of the lock (for each operation, and the average value):

In [20]:
occupancy, occupancy_df = calculate_lock_occupancy(lock_chamber)
display(occupancy_df)
print(f"The mean occupancy equals {occupancy} %")

,leveling_start,leveling_stop,Number of vessels,Lock length,Lock length claimed by vessels,Lock occupancy
0,2025-01-01 01:43:04.354414,2025-01-01 01:53:04.354414,0,400,0.0,0.00
1,2025-01-01 02:25:12.853813,2025-01-01 02:35:12.853813,3,400,300.0,0.75
2,2025-01-01 03:04:55.046038,2025-01-01 03:14:55.046038,4,400,400.0,1.00
3,2025-01-01 03:43:28.814936,2025-01-01 03:53:28.814936,1,400,100.0,0.25
4,2025-01-01 04:14:25.391612,2025-01-01 04:24:25.391612,2,400,200.0,0.50
5,2025-01-01 04:49:44.776060,2025-01-01 04:59:44.776060,3,400,300.0,0.75
6,2025-01-01 05:18:51.166654,2025-01-01 05:28:51.166654,0,400,0.0,0.00
7,2025-01-01 05:54:31.339441,2025-01-01 06:04:31.339441,1,400,100.0,0.25
8,2025-01-01 06:26:50.723889,2025-01-01 06:36:50.723889,3,400,300.0,0.75
9,2025-01-01 07:03:47.300564,2025-01-01 07:13:47.300564,2,400,200.0,0.50


The mean occupancy equals 45.45 %


###### <strong>I/C ratio</strong>
We show the results of the calculation of the capacity of the lock:

In [21]:
capacity, cycle_duration, cycle_duration_in_detail = estimate_lock_capacity(lock_chamber)
print(f"The capacity of the lock is {np.round(capacity,2)} vessels/hour \n")
print(f"The cycle duration equals {cycle_duration['Cycle duration']}, caused by:")
display(pd.Series(cycle_duration).drop("Cycle duration"))
print()
print("The following information was used/estimated to determine the capacity:")
display(cycle_duration_in_detail)

The capacity of the lock is 5.62 vessels/hour 

The cycle duration equals 0 days 01:25:24, caused by:


Entering time (%)     26.58
Operation time (%)    46.83
Exiting time (%)      26.58
dtype: object


The following information was used/estimated to determine the capacity:


Number of vessels in lock                                                            4.0
Sailing distance from crossing point to first gate (first vessel)                  370.0
Sailing speed from crossing point to first gate (first vessel)                         4
Sailing time from crossing point to first gate (first vessel)            0 days 00:01:32
Time gap between vessels sailing in                                      0 days 00:03:00
Total time of vessels sailing in from first vessel until last vessel     0 days 00:09:00
Distance from first lock gate to position in lock (last vessel)                     50.0
Sailing-in speed in lock (last vessel)                                          1.028889
Sailing time from first lock gate to position in lock (last vessel)      0 days 00:00:49
Closing gate time                                                                  300.0
Levelling time                                                                     600.0
Opening gate time    

We can also determine the intensity <i>I_s</i> for each half lock operation cycle

In [22]:
display(Tc_df[['Start time of cycle', 'Stop time of cycle', 'Direction first operation', 'Direction second operation', 
               'Number of upstream vessels', 'Number of downstream vessels', 'Intensity (I_s)']])
print(f"This results in an cycle-average intensity of: {np.round(Tc_df['Intensity (I_s)'].mean(),2)} vessels/hour")

,Start time of cycle,Stop time of cycle,Direction first operation,Direction second operation,Number of upstream vessels,Number of downstream vessels,Intensity (I_s)
0,2025-01-01 01:48:04.354414000,2025-01-01 02:47:01.449925,0,1,0,3,3.053353
1,2025-01-01 02:08:04.354415001,2025-01-01 03:29:43.642151,1,0,4,3,5.143605
2,2025-01-01 02:47:01.449926001,2025-01-01 03:59:17.411049,0,1,4,1,4.151329
3,2025-01-01 03:29:43.642152000,2025-01-01 04:33:13.987723,1,0,2,1,2.834389
4,2025-01-01 03:59:17.411050000,2025-01-01 05:11:33.372173,0,1,2,3,4.151329
5,2025-01-01 04:36:18.987723000,2025-01-01 05:34:38.372173,1,0,0,3,3.086257
6,2025-01-01 05:23:51.166654000,2025-01-01 06:10:19.935553,0,1,0,1,1.290892
7,2025-01-01 05:43:51.166654000,2025-01-01 06:48:39.320002,1,0,3,1,3.703558
8,2025-01-01 06:10:19.935554000,2025-01-01 07:22:35.896675,0,1,3,2,4.151329
9,2025-01-01 06:48:39.320003000,2025-01-01 07:52:09.665575,1,0,1,2,2.834389


This results in an cycle-average intensity of: 3.44 vessels/hour


###### <strong>Aggregated lock performance</strong>
The observed KPIs show that there is some significant delay, as the I/C-ratio is high (0.61). Loop times are significant, particularly due to the duration of the levelling (30.6%) and gate movements (30.6%).

In [23]:
summary = lock_chamber.get_performance()